# (260922) 한국어 이미지 객관식 VQA 개선 베이스라인

원본 `(260827)_baseline_desktop5060ti.ipynb`의 **환경 → 데이터 → 모델 → 프롬프트 → 학습 → 검증 → 제출** 순서를 따릅니다. 이 노트북은 실행 함수를 모두 포함하며 다른 Python 파일을 import하지 않습니다. 기존 노트북은 보존합니다. 측정 근거와 발표용 비교는 `개선_기록.md`에 있습니다.

현재 확인된 로컬 671문항 검증 정확도는 Qwen2.5-VL-3B 0.8823, Qwen3.5-9B 0.9463입니다. 첫 100문항의 0.97은 전체 성능이나 Kaggle 점수가 아닙니다. 대회 평가지표는 Accuracy이며, 공개 Hugging Face VQA 모델·LoRA·양자화 사용이 허용됩니다. 외부 API 추론은 금지됩니다.


## Kaggle GPU 실행 절차

이 노트북 한 파일을 Kaggle Code에 가져오고 대회 데이터를 연결합니다. GPU와 Internet을 켜고 첫 셀에서 `transformers>=5`, `peft`, `bitsandbytes`, `accelerate` 설치 여부를 확인하세요. 설치가 필요하면 커널을 재시작한 뒤 다시 실행합니다. Qwen3.5-9B 공개 모델 가중치를 Hugging Face에서 내려받을 수 있어야 합니다.

설정 셀에서 `RUN_TRAIN=True`, `RUN_EVAL=True`, `RUN_SUBMIT=False`로 먼저 학습과 **671개 전체 검증**을 실행합니다. 결과는 `/kaggle/working/notebook_qwen35_9b/validation.json`, 어댑터는 `adapter/`에 저장됩니다. 검증이 기존 0.94635보다 올라야 다음 실행에서 `RUN_TRAIN=False`, `ADAPTER_PATH`를 저장된 어댑터 경로로 지정하고 `RUN_EVAL=True`, `RUN_SUBMIT=True`로 6,714개 제출 파일을 만듭니다. 새 세션이라면 어댑터가 저장된 Kaggle Dataset을 입력에 연결해야 합니다.

4비트 9B QLoRA는 제한된 GPU에서 실행 가능성을 높이지만, Qwen3.5 개발사 안내에 따르면 양자화 차이가 커 학습 효과가 제한될 수 있습니다. VRAM 24GB 이상을 사용할 수 있으면 `four_bit=False`로 BF16 LoRA를 별도 실험하고 같은 검증 세트로 비교합니다. T4 두 장은 단일 GPU 32GB처럼 합쳐지지 않습니다. 각 실험의 모델, GPU, 학습 단계, 검증 수치, 공개 점수는 `개선_기록.md`에 따로 적습니다.


## 1. 환경 준비

CUDA, PyTorch, Transformers, PEFT, bitsandbytes를 확인합니다. 처음 설치하는 환경은 Qwen3.5 모델 카드의 의존성을 설치하고 커널을 재시작하세요. 로컬 `qwen35_libs`가 있으면 사용합니다.


In [ ]:
# Kaggle의 기본 Transformers가 5.x보다 오래됐으면 Qwen3.5용 패키지를 설치합니다.
# 설치 직후 커널을 한 번 재시작하고 이 노트북을 처음부터 다시 실행하세요.
from pathlib import Path
import importlib.metadata as metadata
import subprocess, sys

if Path("/kaggle/input").is_dir():
    try:
        installed = metadata.version("transformers")
    except metadata.PackageNotFoundError:
        installed = "0"
    required = ("peft", "bitsandbytes", "accelerate")
    missing = []
    for package in required:
        try:
            metadata.version(package)
        except metadata.PackageNotFoundError:
            missing.append(package)
    if int(installed.split(".")[0]) < 5 or missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U",
                               "transformers>=5.17,<6", "peft>=0.21", "bitsandbytes>=0.50",
                               "accelerate>=1.15"])
        raise RuntimeError("패키지 설치 완료: Kaggle 커널을 재시작한 뒤 처음부터 다시 실행하세요.")


In [ ]:
from pathlib import Path
import csv, json, math, random, re, sys
from collections import Counter

# 로컬 저장소와 Kaggle 대회 입력 디렉터리에서 train/test CSV와 이미지를 찾습니다.
KAGGLE = Path("/kaggle/input").is_dir()
if KAGGLE:
    candidates = [p.parent for p in Path("/kaggle/input").rglob("train.csv")
                  if (p.parent / "test.csv").is_file()
                  and (p.parent / "train").is_dir()
                  and (p.parent / "test").is_dir()]
    assert len(candidates) == 1, f"대회 데이터 폴더를 하나로 특정할 수 없습니다: {candidates}"
    DATA = candidates[0]
    ROOT = Path("/kaggle/working")
else:
    ROOT = Path.cwd().resolve()
    if not (ROOT / "data/train.csv").exists():
        ROOT = ROOT.parent
    DATA = ROOT / "data"
assert (DATA / "train.csv").is_file(), f"train.csv를 찾지 못했습니다: {DATA}"
OUTPUT_ROOT = ROOT if KAGGLE else ROOT / "output"

# Qwen3.5에는 Transformers 5.17.0이 필요합니다. 로컬 별도 라이브러리가 있으면 우선 사용합니다.
QWEN35_LIBS = ROOT / "qwen35_libs"
if QWEN35_LIBS.exists() and str(QWEN35_LIBS) not in sys.path:
    sys.path.insert(0, str(QWEN35_LIBS))

import torch, transformers, peft, bitsandbytes
print("root:", ROOT)
print("torch:", torch.__version__, "transformers:", transformers.__version__)
print("GPU:", torch.cuda.get_device_name() if torch.cuda.is_available() else "없음")
assert torch.cuda.is_available(), "4비트 VQA 실행에 CUDA GPU가 필요합니다."


## 2. 실행 설정

검증·학습·제출은 플래그로 켭니다. 기본값은 데이터와 함수만 확인합니다. 16GB GPU에서 9B LoRA 학습은 가속 커널 부재로 느립니다.


In [ ]:
# 모델은 Qwen3.5-9B가 현재 전체 검증에서 가장 좋았습니다.
MODEL_ID = str(ROOT / "downloads/models/Qwen3.5-9B")
if not Path(MODEL_ID).exists():
    MODEL_ID = "Qwen/Qwen3.5-9B"  # 공개 가중치를 다운로드합니다.
ADAPTER_PATH = None              # 학습된 LoRA 어댑터 경로가 있으면 지정
MAX_PIXELS = 1024 * 1024
DECISION = "logits"              # Qwen3.5의 한 글자 답 안정화
BATCH_SIZE = 4                   # 100문항에서 단일 추론과 예측 일치 확인
PROMPT_STYLE = "base"            # "ocr"은 100문항에서 개선되지 않았음
VALID_LIMIT = None               # None: 고정 검증 671문항 전체

RUN_TRAIN = KAGGLE               # Kaggle에서 1,000장 파일럿 실행
RUN_EVAL = KAGGLE                # 학습 후 671문항 전체 검증
RUN_SUBMIT = False               # 검증 결과 확인 후 True로 바꿔 CSV 생성
EPOCHS = 1
MAX_TRAIN_STEPS = 250            # 첫 실험 1,000장; 전체 6,043장은 None
GRAD_ACCUM = 4
LEARNING_RATE = 1e-4
OUTPUT_DIR = OUTPUT_ROOT / "notebook_qwen35_9b"
FOUR_BIT = True                 # 24GB+ GPU의 BF16 비교 실험 시 False
assert not RUN_SUBMIT or RUN_EVAL, "같은 실행에서 검증을 먼저 수행하세요."


## 3. 데이터 로드 및 고정 분할

원본의 200개 표본 대신 학습 CSV 전체를 사용합니다. 시드 42로 6,043개 학습·671개 검증을 만듭니다.


In [ ]:
def read_rows(name: str) -> list[dict[str, str]]:
    with (DATA / f"{name}.csv").open(encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))


def split_train(seed: int = 42, valid_fraction: float = 0.1):
    rows = read_rows("train")
    random.Random(seed).shuffle(rows)
    cut = round(len(rows) * (1 - valid_fraction))
    return rows[:cut], rows[cut:]


train_rows, valid_rows = split_train(seed=42, valid_fraction=0.1)
test_rows = read_rows("test")
print("학습/검증/테스트:", len(train_rows), len(valid_rows), len(test_rows))
print("학습 답 분포:", dict(Counter(row["answer"] for row in train_rows)))
assert (len(train_rows), len(valid_rows), len(test_rows)) == (6043, 671, 6714)


## 4. 객관식 프롬프트

답은 a/b/c/d 중 한 글자입니다. 원본과 달리 생성 결과를 프롬프트와 분리해 파싱합니다.


In [ ]:
CHOICES = "abcd"
def prompt(row: dict[str, str]) -> str:
    prefix = ""
    if PROMPT_STYLE == "ocr":
        prefix = ("이미지의 글자와 숫자를 꼼꼼히 읽고 선택지와 한 글자씩 비교하세요. "
                  "추측하지 말고 사진에 보이는 내용으로 판단하세요.\n")
    return (
        prefix + f"{row['question']}\n"
        f"(a) {row['a']}\n(b) {row['b']}\n"
        f"(c) {row['c']}\n(d) {row['d']}\n\n"
        "정답에 해당하는 알파벳 소문자 한 글자만 답하세요."
    )


def messages(row: dict[str, str], answer: str | None = None,
             views: str = "full"):
    user_content = [{"type": "image"}]
    instruction = prompt(row)
    if views == "grid4":
        user_content.extend({"type": "image"} for _ in range(4))
        instruction = ("첫 이미지는 원본이고 다음 네 이미지는 좌상, 우상, 좌하, 우하 영역입니다. "
                       "모든 이미지의 글자를 확인하세요.\n" + instruction)
    user_content.append({"type": "text", "text": instruction})
    chat = [
        {"role": "system", "content": "이미지를 보고 객관식 문제를 푸세요. 답은 a, b, c, d 중 한 글자입니다."},
        {"role": "user", "content": user_content},
    ]
    if answer is not None:
        chat.append({"role": "assistant", "content": answer})
    return chat


def parse_answer(text: str) -> str | None:
    match = re.fullmatch(r"\s*\(?([abcd])\)?[.。]?\s*", text.lower())
    return match.group(1) if match else None



## 5. 4비트 모델과 이미지 전처리

원본 384² 픽셀 제한을 1MP로 늘립니다. Qwen3.5-9B를 4비트로 로드합니다. 모델 다운로드 또는 로딩에 시간이 걸립니다.


In [ ]:
def load_components(model_id: str, max_pixels: int, adapter: str | None = None,
                    four_bit: bool = True):
    import torch
    import transformers
    from transformers import AutoConfig, AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    processor = AutoProcessor.from_pretrained(model_id, min_pixels=256 * 28 * 28,
                                               max_pixels=max_pixels)
    config = AutoConfig.from_pretrained(model_id)
    model_class = AutoModelForImageTextToText
    if config.model_type == "qwen3_5":
        from transformers import AutoModelForMultimodalLM
        model_class = AutoModelForMultimodalLM
    dtype_argument = "dtype" if int(transformers.__version__.split(".")[0]) >= 5 else "torch_dtype"
    kwargs = {"device_map": "auto", dtype_argument: dtype}
    if four_bit:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype,
        )
    model = model_class.from_pretrained(model_id, **kwargs)
    if adapter:
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, adapter)
    return model, processor


model = processor = None
if RUN_TRAIN or RUN_EVAL or RUN_SUBMIT:
    model, processor = load_components(MODEL_ID, MAX_PIXELS, ADAPTER_PATH, four_bit=FOUR_BIT)
    print("모델 준비 완료:", MODEL_ID)


## 6. 이미지 입력과 추론

선택지 토큰 점수로 답을 결정합니다. 4문항 배치 추론은 첫 100문항에서 단일 추론과 결과가 같았습니다.


In [ ]:
def make_inputs(processor, row, answer: str | None = None,
                views: str = "full"):
    from PIL import Image

    image_path = DATA / row["path"]
    with Image.open(image_path) as source:
        image = source.convert("RGB")
    images = [image]
    if views == "grid4":
        width, height = image.size
        x_centers = [width * 0.275, width * 0.725]
        y_centers = [height * 0.275, height * 0.725]
        crop_width, crop_height = width * 0.55, height * 0.55
        for cy in y_centers:
            for cx in x_centers:
                box = (round(cx - crop_width / 2), round(cy - crop_height / 2),
                       round(cx + crop_width / 2), round(cy + crop_height / 2))
                images.append(image.crop(box))
    chat = messages(row, answer, views)
    text = processor.apply_chat_template(
        chat, tokenize=False, add_generation_prompt=answer is None,
        enable_thinking=False,
    )
    inputs = processor(text=[text], images=images, return_tensors="pt")
    return inputs, image


def predict(model, processor, row, max_new_tokens: int = 4,
            decision: str = "generate", views: str = "full"):
    import torch

    inputs, _ = make_inputs(processor, row, views=views)
    inputs = inputs.to(model.device)
    if decision == "logits":
        token_ids = [processor.tokenizer.encode(letter, add_special_tokens=False)
                     for letter in CHOICES]
        if any(len(ids) != 1 for ids in token_ids):
            raise ValueError("Choice labels are not single tokens")
        with torch.inference_mode():
            logits = model(**inputs).logits[0, -1]
        scores = [float(logits[ids[0]]) for ids in token_ids]
        answer = CHOICES[max(range(4), key=lambda i: scores[i])]
        return answer, json.dumps(dict(zip(CHOICES, scores)))
    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                 do_sample=False)
    new_ids = outputs[:, inputs["input_ids"].shape[1]:]
    response = processor.batch_decode(new_ids, skip_special_tokens=True)[0]
    return parse_answer(response), response


def predict_logits_batch(model, processor, rows):
    """Score four answer tokens in one forward pass for several full-view images."""
    import torch
    from PIL import Image

    texts, images = [], []
    for row in rows:
        texts.append(processor.apply_chat_template(
            messages(row), tokenize=False, add_generation_prompt=True,
            enable_thinking=False,
        ))
        with Image.open(DATA / row["path"]) as source:
            images.append(source.convert("RGB"))
    previous_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "left"
    try:
        inputs = processor(text=texts, images=images, padding=True,
                           return_tensors="pt").to(model.device)
    finally:
        processor.tokenizer.padding_side = previous_padding_side
    token_ids = [processor.tokenizer.encode(letter, add_special_tokens=False)[0]
                 for letter in CHOICES]
    with torch.inference_mode():
        scores = model(**inputs).logits[:, -1, token_ids].float().cpu().tolist()
    return [(CHOICES[max(range(4), key=lambda i: row_scores[i])],
             json.dumps(dict(zip(CHOICES, row_scores)))) for row_scores in scores]



## 7. LoRA 학습

프롬프트와 패딩을 `-100`으로 가리고 답 토큰만 학습합니다. `RUN_TRAIN=True`일 때 실행합니다. 1단계 점검 후 전체 학습 시간을 확인하세요.


In [ ]:
def train(model, processor, rows, output_dir: Path, epochs: int, max_steps: int | None,
          grad_accum: int, learning_rate: float):
    import torch
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from tqdm.auto import tqdm
    from transformers import get_linear_schedule_with_warmup

    model = prepare_model_for_kbit_training(model)
    model.gradient_checkpointing_enable()
    model.config.use_cache = False
    model = get_peft_model(model, LoraConfig(
        r=8, lora_alpha=16, lora_dropout=0.05, bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        task_type="CAUSAL_LM",
    ))
    model.print_trainable_parameters()
    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),
                                  lr=learning_rate)
    total_batches = len(rows) * epochs
    if max_steps is not None:
        total_batches = min(total_batches, max_steps * grad_accum)
    total_steps = math.ceil(total_batches / grad_accum)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, max(1, round(total_steps * 0.03)), total_steps,
    )
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    scaler = torch.amp.GradScaler("cuda", enabled=dtype == torch.float16)
    batch_count = 0
    optimizer.zero_grad(set_to_none=True)
    model.train()
    for epoch in range(epochs):
        order = rows[:]
        random.Random(42 + epoch).shuffle(order)
        for row in tqdm(order, desc=f"Train epoch {epoch + 1}"):
            full, image = make_inputs(processor, row, row["answer"])
            prefix = processor.apply_chat_template(
                messages(row), tokenize=False, add_generation_prompt=True,
                enable_thinking=False,
            )
            prefix_inputs = processor(text=[prefix], images=[image], return_tensors="pt")
            prompt_length = int(prefix_inputs["attention_mask"][0].sum())
            labels = full["input_ids"].clone()
            labels[full["attention_mask"] == 0] = -100
            labels[:, :prompt_length] = -100
            if not (labels != -100).any():
                raise ValueError(f"No supervised answer tokens: {row['id']}")
            full["labels"] = labels
            full = full.to(model.device)
            with torch.autocast("cuda", dtype=dtype):
                loss = model(**full).loss / grad_accum
            scaler.scale(loss).backward()
            batch_count += 1
            if batch_count % grad_accum == 0 or batch_count == total_batches:
                if not any(p.grad is not None and bool(torch.isfinite(p.grad).all())
                           and bool((p.grad != 0).any())
                           for p in model.parameters() if p.requires_grad):
                    raise RuntimeError("No nonzero finite trainable gradients")
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
            if batch_count >= total_batches:
                break
        if batch_count >= total_batches:
            break
    output_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(output_dir)
    processor.save_pretrained(output_dir)
    print(f"Saved adapter: {output_dir}; batches={batch_count}")
    return model


if RUN_TRAIN:
    model = train(model, processor, train_rows, OUTPUT_DIR / "adapter", EPOCHS,
                  MAX_TRAIN_STEPS, GRAD_ACCUM, LEARNING_RATE)


## 8. 671문항 정확도 검증

학습 손실 대신 객관식 정확도와 답 파싱 실패를 기록합니다. 점수 비교에는 `VALID_LIMIT=None`을 유지합니다.


In [ ]:
def evaluate(model, processor, rows, limit: int | None, result_path: Path,
             decision: str, views: str, batch_size: int, max_new_tokens: int):
    from tqdm.auto import tqdm

    model.eval()
    selected = rows if limit is None else rows[:limit]
    records = []
    if batch_size > 1 and decision == "logits" and views == "full":
        groups = [selected[i:i + batch_size]
                  for i in range(0, len(selected), batch_size)]
        for group in tqdm(groups, desc="Validate batches"):
            for row, (prediction, raw) in zip(
                    group, predict_logits_batch(model, processor, group)):
                records.append({"id": row["id"], "gold": row["answer"],
                                "prediction": prediction, "raw": raw,
                                "correct": prediction == row["answer"]})
    else:
        for row in tqdm(selected, desc="Validate"):
            prediction, raw = predict(model, processor, row, decision=decision,
                                      views=views, max_new_tokens=max_new_tokens)
            records.append({"id": row["id"], "gold": row["answer"],
                            "prediction": prediction, "raw": raw,
                            "correct": prediction == row["answer"]})
    accuracy = sum(r["correct"] for r in records) / len(records)
    invalid = sum(r["prediction"] is None for r in records)
    result_path.parent.mkdir(parents=True, exist_ok=True)
    result_path.write_text(json.dumps({"count": len(records), "accuracy": accuracy,
                                      "invalid": invalid, "records": records},
                                     ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps({"count": len(records), "accuracy": accuracy,
                      "invalid": invalid, "output": str(result_path)}, ensure_ascii=False))


if RUN_EVAL:
    evaluate(model, processor, valid_rows, VALID_LIMIT,
             OUTPUT_DIR / "validation.json", DECISION, "full", BATCH_SIZE, 4)
    result = json.loads((OUTPUT_DIR / "validation.json").read_text(encoding="utf-8"))
    print("검증 정확도:", result["accuracy"], "파싱 실패:", result["invalid"])
    assert result["invalid"] == 0


## 9. 제출 CSV 생성

검증 결과를 확인한 뒤 `RUN_SUBMIT=True`로 실행합니다. `id,answer` 형식으로 6,714행을 저장하고 중단 시 `.partial.csv`에서 재개합니다. Kaggle 공개 점수는 실제 제출 후에만 기록합니다.


In [ ]:
def submit(model, processor, output_path: Path, decision: str, views: str,
           batch_size: int, max_new_tokens: int):
    from tqdm.auto import tqdm

    model.eval()
    rows = read_rows("test")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    partial_path = output_path.with_suffix(".partial.csv")
    completed = 0
    if partial_path.exists():
        with partial_path.open(encoding="utf-8", newline="") as f:
            previous = list(csv.DictReader(f))
        if [r["id"] for r in previous] != [r["id"] for r in rows[:len(previous)]]:
            raise ValueError("Partial submission IDs do not match test order")
        if any(r["answer"] not in CHOICES for r in previous):
            raise ValueError("Partial submission contains invalid answers")
        completed = len(previous)
    with partial_path.open("a", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["id", "answer"])
        if completed == 0:
            writer.writeheader()
        remaining = rows[completed:]
        if batch_size > 1 and decision == "logits" and views == "full":
            groups = [remaining[i:i + batch_size]
                      for i in range(0, len(remaining), batch_size)]
            for group in tqdm(groups, desc="Test batches"):
                for row, (answer, raw) in zip(
                        group, predict_logits_batch(model, processor, group)):
                    writer.writerow({"id": row["id"], "answer": answer})
                f.flush()
        else:
            for row in tqdm(remaining, desc="Test inference"):
                answer, raw = predict(model, processor, row, decision=decision,
                                      views=views, max_new_tokens=max_new_tokens)
                if answer is None:
                    raise ValueError(f"Cannot parse model response for {row['id']}: {raw!r}")
                writer.writerow({"id": row["id"], "answer": answer})
                f.flush()
    partial_path.replace(output_path)
    print(f"Saved {len(rows)} rows: {output_path}")


if RUN_SUBMIT:
    submit(model, processor, OUTPUT_DIR / "submission.csv", DECISION, "full",
           BATCH_SIZE, 4)
    with (OUTPUT_DIR / "submission.csv").open(encoding="utf-8", newline="") as f:
        submission_rows = list(csv.DictReader(f))
    assert [r["id"] for r in submission_rows] == [r["id"] for r in test_rows]
    assert all(r["answer"] in CHOICES for r in submission_rows)
    print("제출 CSV 확인:", OUTPUT_DIR / "submission.csv", len(submission_rows))


## 10. 발표용 오류 분석

선택 사항입니다. 이미지 크기와 이미 저장한 두 모델의 검증 예측을 대조합니다. 이 분석에는 테스트 정답을 사용하지 않습니다.


In [ ]:
# 발표용 데이터 확인. 필요할 때만 실행합니다.
RUN_IMAGE_STATS = False
RUN_ENSEMBLE_ANALYSIS = False
if RUN_IMAGE_STATS:
    from PIL import Image
    sizes = Counter()
    for row in read_rows("train"):
        with Image.open(DATA / row["path"]) as image:
            sizes[image.size] += 1
    print("1MP 초과:", sum(count for (w, h), count in sizes.items()
                          if w * h > 1024 * 1024), "/", sum(sizes.values()))

if RUN_ENSEMBLE_ANALYSIS:
    path_9b = ROOT / "output/qwen35_9b_logits_full/validation.json"
    path_25 = ROOT / "output/qwen25_1mp_full/validation.json"
    rows_9b = json.loads(path_9b.read_text(encoding="utf-8"))["records"]
    rows_25 = {r["id"]: r for r in
               json.loads(path_25.read_text(encoding="utf-8"))["records"]}
    for threshold in (0, 0.5, 1.0, 2.0):
        correct = 0
        for row in rows_9b:
            values = sorted(json.loads(row["raw"]).values(), reverse=True)
            use_25 = values[0] - values[1] < threshold
            prediction = rows_25[row["id"]]["prediction"] if use_25 else row["prediction"]
            correct += prediction == row["gold"]
        print("threshold", threshold, "accuracy", correct / len(rows_9b))


## 실험 기록

대회 원본 공개 점수는 0.70837, 팀의 별도 Qwen3.5-9B 제출은 0.95829입니다. 이 노트북의 9B 설정은 로컬 671문항에서 0.9463을 기록했습니다. 차이와 실패한 OCR·확대 실험은 `개선_기록.md`에 남깁니다.
